# River Plastic Ranking — Feature Matrix rebuild (Google Colab)

Rebuilds `data/processed/feature_matrix_v1.csv`, which needs the 1.5 GB HydroRIVERS
geodatabase and **>8 GB RAM** (it OOMs on smaller machines). Colab's free tier (~12 GB)
handles it.

**Run cells top to bottom.** Runtime → *(optional)* High-RAM if you have Colab Pro.

Steps: install deps → clone repo → download raw data from Zenodo (2.86 GB) →
extract VIIRS nightlights (missing intermediate) → run notebook 02 → download the result.

*Constructed from the verified local pipeline; run cell-by-cell and check each output.*

In [ ]:
# 1. Dependencies (pinned to the working versions)
!pip install -q geopandas==1.1.4 pyogrio==0.13.0 shapely==2.1.2 pyproj==3.7.1 \
    rasterio==1.4.4 xarray==2025.6.1 openpyxl==3.1.5 nbconvert nbclient ipykernel
print("deps installed")

In [ ]:
# 2. Clone the repo
import os
if not os.path.isdir("river-plastic-ranking"):
    !git clone --depth 1 https://github.com/pablocs116/river-plastic-ranking.git
%cd river-plastic-ranking
!git log --oneline -1

In [ ]:
# 3. Recover raw inputs from Zenodo (2.86 GB) and unpack into data/raw/
#    DOI 10.5281/zenodo.20793131 (CC BY 4.0)
import os, hashlib, subprocess
ZURL = "https://zenodo.org/api/records/20793131/files/river-plastic-ranking-raw-data.zip/content"
ZMD5 = "d362c319bad2a2e43853e0993455e086"
zip_path = "data/raw/river-plastic-ranking-raw-data.zip"
os.makedirs("data/raw", exist_ok=True)
if not os.path.exists("data/raw/meijer2021"):
    if not os.path.exists(zip_path):
        subprocess.run(["wget", "-q", "-O", zip_path, ZURL], check=True)
    # verify checksum
    h = hashlib.md5()
    with open(zip_path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    assert h.hexdigest() == ZMD5, f"MD5 mismatch: {h.hexdigest()}"
    print("MD5 OK — unpacking...")
    subprocess.run(["unzip", "-nq", zip_path, "-d", "data/raw/"], check=True)
print("raw data ready:", sorted(os.listdir("data/raw")))

In [ ]:
# 4. Extract VIIRS nightlights at the 31,819 outfalls (missing intermediate that
#    notebook 02 expects at data/processed/viirs_nightlight_at_outfalls.csv).
import gzip, shutil
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio

RAW = Path("data/raw"); PROC = Path("data/processed"); PROC.mkdir(exist_ok=True)
gz = RAW / "viirs/VNL_v21_npp_2021_global_vcmslcfg_c202205302300.median_masked.dat.tif.gz"
tif = gz.with_suffix("")          # drop .gz
if not tif.exists():
    print("gunzip VIIRS raster...")
    with gzip.open(gz, "rb") as fi, open(tif, "wb") as fo:
        shutil.copyfileobj(fi, fo)

shp = gpd.read_file(RAW / "meijer2021/Meijer2021_midpoint_emissions.shp").to_crs(4326)
lons, lats = shp.geometry.x.values, shp.geometry.y.values
with rasterio.open(tif) as src:
    vals = np.array([v[0] for v in src.sample(zip(lons, lats))], dtype=float)
    if src.nodata is not None:
        vals[vals == src.nodata] = np.nan
out = PROC / "viirs_nightlight_at_outfalls.csv"
pd.DataFrame({"lon": lons, "lat": lats, "nightlight_intensity": vals}).to_csv(out, index=False)
print(f"VIIRS extracted: {np.isfinite(vals).sum():,}/{len(vals):,} valid -> {out}")

In [ ]:
# 5. Run notebook 02 (kernel override: the notebook hard-codes a 'riverplastic' kernel).
#    nbconvert runs it in notebooks/, so its ../data/raw paths resolve correctly.
!python -m nbconvert --to notebook --execute \
    --ExecutePreprocessor.timeout=3600 \
    --ExecutePreprocessor.kernel_name=python3 \
    --output 02_feature_engineering_executed \
    notebooks/02_feature_engineering.ipynb
print("done")

In [ ]:
# 6. Verify and download the rebuilt feature matrix
import pandas as pd
fm = pd.read_csv("data/processed/feature_matrix_v1.csv")
print("feature_matrix_v1.csv:", fm.shape)
print(list(fm.columns))
from google.colab import files
files.download("data/processed/feature_matrix_v1.csv")

## After downloading

Put `feature_matrix_v1.csv` back into `data/processed/` on your machine, then notebooks
04/06/07/08 (calibration, ranking, plastic+nightlight model) can run locally — they are
not memory-heavy. That also unblocks re-verifying the Japan-sensitivity calibration
(slope 0.71 → 0.98) once `observed_flux_matched_S3.csv` is reconstructed.